# SmartFood · Baseline por perfil de usuario

Recomendar recetas no evaluadas de la categoría más frecuente del usuario, ordenadas por cercanía calórica.


**Estado:** ejecución completa mediante el maestro el 6 de octubre de 2026. Consulta el [registro de ejecución](../docs/ejecuciones.md) para el detalle por etapa.

**Archivo:** `03_baseline_perfil_usuario.ipynb`. Consulta la [guía de ejecución](../docs/notebooks.md) para conocer entradas, salidas y precauciones.


In [1]:
# Resolver la raíz del proyecto desde Jupyter o desde la carpeta del proyecto.
from pathlib import Path
import os
import sys

def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("smartfood")
if PROJECT_ROOT is None:
    raise RuntimeError("Inicia Jupyter desde la carpeta SmartFood o notebooks.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/intermediate", "data/processed", "data/output", "reports", "models"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Configuracion comun: registrar avisos para el resumen de alertas.
from smartfood.entorno_notebook import configurar_notebook
configurar_notebook()


## ✅ Paso 1: Carga de librerías

In [2]:
import pandas as pd
from smartfood.recomendaciones_base import recomendar_por_perfil_usuario


## ✅ Paso 2: Carga de datos procesados

In [3]:
recipes = pd.read_parquet("data/processed/recipes_filtrado.parquet")
reviews = pd.read_parquet("data/processed/reviews_filtrado.parquet")
print("Recipes:", recipes.shape)
print("Reviews:", reviews.shape)

Recipes: (190, 28)
Reviews: (10343, 8)


## ✅ Paso 3: Recomendacion Por perfil de usuario

**Nota sobre la cantidad de recomendaciones:** `top_n` indica el número máximo de recetas a devolver; no garantiza obtener esa cantidad. El resultado depende del catálogo disponible y del historial de cada usuario. La función excluye las recetas ya evaluadas, descarta candidatos sin calorías, categoría o nombre y conserva únicamente los de la categoría más frecuente del historial. Si después de estos filtros quedan menos de `top_n` candidatos, devuelve solo los disponibles; si no queda ninguno, el resultado está vacío. También devuelve un resultado vacío si no encuentra recetas del historial del usuario en el catálogo.

En este baseline, la distancia calórica ordena los candidatos, pero no los elimina: no se aplica un umbral máximo de diferencia. Por eso, aumentar `top_n` no amplía las categorías permitidas ni recupera recetas excluidas.


In [4]:
usuario_ejemplo = reviews["authorid"].iloc[40]
nombre_usuario =reviews.loc[reviews["authorid"] == usuario_ejemplo, "authorname"].values[0]
print(f"Usuario de ejemplo: {nombre_usuario} ({usuario_ejemplo})")
top_n = 5
recomendaciones = recomendar_por_perfil_usuario(usuario_ejemplo, reviews, recipes, top_n=top_n)
recomendaciones


Usuario de ejemplo: kim d (198)

📌 Recomendaciones para el usuario 198 (calorías ~430, categoría: dessert):
189. microwave chocolate mug brownie (442.6 cal)
142. ednas apple crumble aka apple crisp (323.8 cal)
50. chocolate chocolate chocolate bundt cake with chocolate glaze (321.6 cal)
186. vanilla buttercream frosting from sprinkles cupcakes (273.1 cal)
19. lemon bars (592.2 cal)


,recipeid,name,authorid,authorname,cooktime,preptime,totaltime,datepublished,description,images,...,cholesterolcontent,sodiumcontent,carbohydratecontent,fibercontent,sugarcontent,proteincontent,recipeservings,recipeyield,recipeinstructions,calorias_diff
0,97,microwave chocolate mug brownie,642765,wheres_the_beef,pt1m,pt5m,pt6m,2009-01-11 22:20:00+00:00,this makes a delicious brownie for one in abou...,[https://img.sndimg.com/food/image/upload/w_55...,...,61.1,361.9,55.4,4.4,25.6,5.6,1.0,1 brownie,"[In a coffee mug, add water/milk, melted butte...",13.047059
1,122,ednas apple crumble aka apple crisp,15718,steve p,pt35m,pt10m,pt45m,2004-02-03 20:00:00+00:00,a simple and easy to make but oh so yummy reci...,[https://img.sndimg.com/food/image/upload/w_55...,...,27.1,99.6,56.7,4.1,39.5,2.5,NaN,None,[Put apples in an 8 or 9 inch square baking di...,105.752941
2,102,chocolate chocolate chocolate bundt cake with ...,24235,ccinsc,pt50m,pt15m,pt1h5m,2002-05-03 10:49:00+00:00,need i say morechocolate decadently rich and m...,[https://img.sndimg.com/food/image/upload/w_55...,...,27.1,393.3,51.4,2.5,36.0,3.9,16.0,1 cake,"[Cake: Preheat oven to 350°., Grease and flour...",107.952941
3,168,vanilla buttercream frosting from sprinkles cu...,454328,c taylor,None,pt10m,pt10m,2007-04-12 20:30:00+00:00,my sister got this buttercream frosting off th...,[https://img.sndimg.com/food/image/upload/w_55...,...,40.7,27.2,35.0,0.0,34.3,0.2,NaN,12 cupcakes,[In a bowl add sugar then add butter. Beat til...,156.452941
4,176,lemon bars,10404,diana adcock,pt50m,pt20m,pt1h10m,2001-07-05 10:15:00+00:00,make and share this lemon bars recipe from foo...,[https://img.sndimg.com/food/image/upload/w_55...,...,154.0,262.2,85.3,1.0,57.7,7.0,NaN,12 bars or so,[For the base mix the butter into the flour an...,162.647059


In [5]:
from IPython.display import Markdown, display
from smartfood.anotaciones import comentario_recomendaciones
display(Markdown(comentario_recomendaciones(recomendaciones, top_n)))

from smartfood.informe import registrar_etapa
registrar_etapa(PROJECT_ROOT, "baseline", {"top_n_solicitado": int(top_n), "recetas_devueltas": len(recomendaciones)}, comentario_recomendaciones(recomendaciones, top_n), "03_baseline_perfil_usuario.ipynb")


### Resultado del baseline

Se solicitaron **5** recetas y se devolvieron **5**. La lista alcanza la cantidad solicitada. En este baseline se excluyen recetas vistas, candidatos con datos requeridos faltantes y categorías distintas de la más frecuente. La distancia calórica solo ordena; completar la lista no demuestra relevancia ni calidad nutricional.

## Alertas de la ejecución

La tabla agrupa los avisos de Python y conserva su detalle original. Si una celda falla antes de llegar aquí, puedes ejecutar esta celda de resumen para consultar las alertas ya registradas. Los errores y los mensajes de diagnóstico impresos no se incluyen automáticamente.


In [6]:
from smartfood.entorno_notebook import mostrar_alertas
mostrar_alertas(PROJECT_ROOT / "reports" / "alertas" / "03_baseline_perfil_usuario.json")
from smartfood.informe import generar_informe
generar_informe(PROJECT_ROOT)


WindowsPath('D:/portafolio/Analisis/Recommenders/SmartFood/reports/informe_final/informe_final.md')